In [183]:
from unstructured.partition.pdf import partition_pdf
from IPython.display import Image, display
import base64
import pandas as pd
import io
import tabulate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from collections import defaultdict
import json
import psycopg2
from pgvector.psycopg2 import register_vector
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.document_loaders import PyPDFLoader
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from ragas import RunConfig



C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_5600\1446991898.py:23: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_5600\1446991898.py:23: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_5600\1446991898.py:23: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Exam

In [2]:
load_dotenv()

True

In [3]:
# gemini embedding 2 preview is layout aware and can recognise table markdown as a relational grid. 
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview"
)

In [216]:
llm = ChatOllama(model="mistral-small:24b")

In [98]:
llm1 = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [4]:
chunks = partition_pdf(
    filename="government-data-security-policies.pdf",
    strategy="hi_res", # table detection and extraction as text_as_html

    infer_table_structure=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=True, # extracts images and tables as base64 encoded strings

    chunking_strategy="by_title",
    max_characters=2000,
    combine_text_under_n_chars=1000,
    new_after_n_chars=1500
)
'''chunk by title chunks by section headings and titles. For structured documents, each document
element is already semantically sound. computaionally cheaper than semantic chunking which is more 
suitable for highly unstructured data. 
'''

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 14830.14it/s]


'chunk by title chunks by section headings and titles. For structured documents, each document\nelement is already semantically sound. computaionally cheaper than semantic chunking which is more \nsuitable for highly unstructured data. \n'

In [ ]:
string = chunks[20].metadata.orig_elements[0].metadata.image_base64
display(Image(data=base64.b64decode(string)))

In [6]:
'''loops through each element in chunk. If element is table, converts text_as_html to markdown
which is generally better (lower token cost, more readable to llms)
'''
def modify_text(chunk):
    text = []
    for element in chunk.metadata.orig_elements:
        if element.category == "Table" and getattr(element.metadata, "text_as_html", None):
            df = pd.read_html(io.StringIO(element.metadata.text_as_html))[0]
            markdown = df.to_markdown(index=False)
            text.append(markdown)
        else:
            text.append(element.text)
    chunk.text = "\n\n".join(text)

In [7]:
def extract_coords(chunk):
    highlights = []
    orig_elements = chunk.metadata.orig_elements
    for element in orig_elements:
        page = element.metadata.page_number
        coords = element.metadata.coordinates.points

        width = element.metadata.coordinates.system.width
        height = element.metadata.coordinates.system.height

        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
    
        clean_box = [
            round(max(0, float(min(xs))),2),
            round(max(0, float(min(ys))),2),
            round(min(width, float(max(xs))), 2),
            round(min(height, float(max(ys))), 2)
        ]
        highlights.append({
            "page": int(page),
            "bbox": clean_box,
            "width": int(width),
            "height": int(height)
        })
    setattr(chunk.metadata, "highlights", highlights)

In [8]:
def embed_text(chunk):
    vector = embeddings.embed_query(chunk.text)
    setattr(chunk.metadata, "vector", vector)

In [44]:
def number_chunk(chunk, number):
    setattr(chunk.metadata, "chunk_number", number)


In [46]:
def process_chunk(chunk, number):
    modify_text(chunk)
    embed_text(chunk)
    extract_coords(chunk)
    number_chunk(chunk, number)

In [47]:
for index, chunk in enumerate(chunks):
    process_chunk(chunk, index + 1)

In [11]:
import fitz
def visualize_chunk_highlights(pdf_path, chunk, output_path="preview.pdf"):
    doc = fitz.open(pdf_path)
    highlights = getattr(chunk.metadata, "highlights", [])

    for h in highlights:
        page_index = h['page'] - 1
        page = doc[page_index]
        
        # 1. Get the page size as PyMuPDF sees it (72 DPI)
        actual_w = page.rect.width
        actual_h = page.rect.height
        
        # 2. Get the original system size from your metadata
        orig_w = h['width']
        orig_h = h['height']
        
        # 3. SCALE the coordinates
        # Math: (Raw_Value / Original_System_Size) * Actual_PDF_Size
        raw_bbox = h['bbox'] # [x1, y1, x2, y2]
        
        scaled_bbox = [
            (raw_bbox[0] / orig_w) * actual_w, # x_min
            (raw_bbox[1] / orig_h) * actual_h, # y_min
            (raw_bbox[2] / orig_w) * actual_w, # x_max
            (raw_bbox[3] / orig_h) * actual_h  # y_max
        ]
        
        # 4. Draw the correctly scaled rectangle
        rect = fitz.Rect(scaled_bbox)
        page.draw_rect(rect, color=(1, 1, 0), fill=(1, 1, 0), overlay=True, width=0)

    doc.save(output_path)
visualize_chunk_highlights("government-data-security-policies.pdf", chunks[1], output_path="preview.pdf")

In [12]:
print(chunks[20].text)

| 0                       | 1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
|:------------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
chunks[20].metadata.highlights

[{'page': 22,
  'bbox': [234.95, 292.45, 2614.08, 3050.35],
  'width': 2894,
  'height': 4093}]

In [52]:
chunks[20].metadata.chunk_number

21

In [202]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [15]:
chunk = chunks[0]
chunk_embedding = chunk.metadata.vector
chunk_text = chunk.text
if chunk.metadata.image_base64:
    image_base64 = base64.b64decode(chunk.metadata.image_base64)
else: 
    image_base64 = None
highlights = chunk.metadata.highlights

In [16]:
try:
    cur.execute('''
        INSERT INTO chunk_store (chunk_embedding, chunk_text, image_base64, highlights)
        VALUES (%s, %s, %s, %s)
    ''', (chunk_embedding, chunk_text, image_base64, json.dumps(highlights)))
    conn.commit()
except Exception as e: 
    print(f"Postgres Error: {e}")
    conn.rollback()

In [53]:
def store_chunk(chunk):
    chunk_embedding = chunk.metadata.vector
    chunk_text = chunk.text
    if chunk.metadata.image_base64:
        image_base64 = base64.b64decode(chunk.metadata.image_base64)
    else: 
        image_base64 = None
    highlights = chunk.metadata.highlights
    chunk_number = chunk.metadata.chunk_number
    try:
        cur.execute('''
            INSERT INTO chunk_store (chunk_embedding, chunk_text, image_base64, highlights, chunk_number)
            VALUES (%s, %s, %s, %s, %s)
        ''', (chunk_embedding, chunk_text, image_base64, json.dumps(highlights), chunk_number))
        conn.commit()
    except: 
        print("Error")
        conn.rollback()

In [56]:
for chunk in chunks:
    store_chunk(chunk)

In [20]:
cur.execute("SELECT chunk_text FROM chunk_store WHERE id = 22")
data = cur.fetchall()
print(data[0][0])

| 0                       | 1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
|:------------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------

In [188]:
query = "data security risk management"

In [189]:
def search_chunks(query):
    query_vector = embeddings.embed_query(query)
    search_query = '''
        SELECT chunk_number, chunk_text, 1 - (chunk_embedding <=> %s) AS similarity
        FROM chunk_store
        ORDER BY chunk_embedding <=> %s
        LIMIT 5;
    '''
    try:
        cur.execute(search_query, (str(query_vector), str(query_vector)))
        data = cur.fetchall()
        df = pd.DataFrame(data, columns=["chunk_number", "chunk_text", "similarity"])
        return df
    except Exception as e:
        print(f"An error occurred: {e}")
        conn.rollback()
    

In [190]:
df = search_chunks(query)

In [191]:
df.head()


,chunk_number,chunk_text,similarity
0,4,Section 2: Technical and process measures to p...,0.712063
1,2,The Government takes its responsibility as a c...,0.696333
2,9,Government Data Security Policies | 8\n\n\n\nS...,0.695529
3,3,Government Data Security Policies | 2\n\n\n\nS...,0.689794
4,17,\n\nTechnical measures and the complementary p...,0.675025


In [192]:
lis = df["chunk_text"].tolist()
lis

['Section 2: Technical and process measures to protect data and prevent data compromises\n\nAgencies should implement the appropriate technical and process measures to protect 05 data against data security threats and prevent compromises of the conﬁdentiality and integrity of the data.\n\nAgencies should adopt a risk-based approach when implementing the technical and process measures. Technical measures are only eﬀective when the complementary process measures are implemented too (See Annex A).\n\nThe data security technical and process measures are to be implemented on top of any cybersecurity measures in place.\n\nData should be protected according to the risk level determined through a data security risk assessment. Agencies should ensure that data is adequately protected while minimising the impact to operations, resources and costs.\n\n06 Agencies should adopt the following strategies when implementing the technical measures and process measures to safeguard data:\n\na) Reduce the

In [193]:
cur.close()
conn.close()

In [194]:
def generate_context(dataframe):
    context_text = "\n\n".join(
        f"--- Chunk {row['chunk_number']} ---\n{row['chunk_text']}"
        for _, row in dataframe.iterrows()
    )
    return context_text

In [195]:
context_text = generate_context(df)

In [196]:
def generate_message(query, context_text):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context chunks.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. Do not use external knowledge.
    2. CITATIONS: You MUST cite the specific Chunk Number (e.g. [Chunk 4]) for every fact or claim you make. Each chunk is labelled at the start (e.g. --- Chunk 4 ---).
    3. CONTEXT INTEGRATION: Treat the chunks as a single unified knowledge base, even if they are provided out of chronological order.
    4. RELEVANCE: Only use information from chunks that are relevant to answering the question.  
    5. TABLES: If context contains Markdown tables, interpret row-to-column relationships strictly to ensure data accuracy.
    6. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    HumanMessage(content=f"""
    Context: 
    {context_text}

    Question: {query}
    """),

    ]
    return messages

In [197]:
messages = generate_message(query, context_text)

In [198]:
print(messages)

[SystemMessage(content='\n    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context chunks.\n\n    RULES:\n    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. Do not use external knowledge.\n    2. CITATIONS: You MUST cite the specific Chunk Number (e.g. [Chunk 4]) for every fact or claim you make. Each chunk is labelled at the start (e.g. --- Chunk 4 ---).\n    3. CONTEXT INTEGRATION: Treat the chunks as a single unified knowledge base, even if they are provided out of chronological order.\n    4. RELEVANCE: Only use information from chunks that are relevant to answering the question.  \n    5. TABLES: If context contains Markdown tables, interpret row-to-column relationships strictly to ensure data accuracy.\n    6. FORMATTING: Use clear headings and bullet points for complex answers.                          \n    ', additional_kwargs={}, response_

In [199]:
response = llm.invoke(messages)

In [200]:
print(response.content)

**Data Security Risk Management**

To ensure adequate and effective data security risk management, agencies should perform data security risk assessments for their datasets as part of the Government ICT Risk Management Methodology [Chunk 1].

Agencies should use the Data Security Risk Assessment Methodology to conduct data security risk assessments when acquiring a new dataset, developing a new ICT system that contains personal or entity data, or when existing data which has not been risk assessed is first used [Chunks 2-3].

The technical measures and complementary process measures listed in Annex A include:

* Volume limited and time limited data access
* Log and monitor data transactions to detect high-risk or suspicious activity
* Digital watermarking of file
* Email data protection tools
* Data Loss Protection tools
* Hashing with Salt
* Field Level Encryption
* Tokenisation
* Obfuscation/masking/removal of entity attributes
* Dataset partitioning

Agencies should select the measu

In [129]:
ragas_llm = LangchainLLMWrapper(ChatOllama(
    model="llama3-groq-tool-use",
    format="json",
    temperature=0,
    num_ctx=8192
))
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_5600\4180710712.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOllama(
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_5600\4180710712.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [130]:
generator = TestsetGenerator(
    llm=ragas_llm,
    embedding_model=ragas_embeddings
)

In [131]:
loader = PyPDFLoader("government-data-security-policies.pdf")
documents = loader.load()

Ignoring wrong pointing object 8 0 (offset 0)


In [132]:
testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=10
)
test_df = testset.to_pandas()

Generating Samples: 100%|██████████| 12/12 [00:29<00:00,  2.43s/it]


In [219]:
test_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What ar the goverment datasecurity policys?,[Government Data Security Policies |  1\nGO...,This document contains general information for...,Security Consultant,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,What are the key policies for government data ...,[The Government takes its responsibility as a ...,The key policies for government data security ...,Data Security Officer,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
2,What is the Government ICT Risk Management Met...,[Section 1: Data Security Risk Management\n01 ...,The Government ICT Risk Management Methodology...,Data Security Officer,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
3,What are the cybersecurity measures recommende...,[Section 2: Technical and process measures \nt...,Agencies should implement the appropriate tech...,Data Security Officer,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,What measures can be implemented to mitigate d...,[<1-hop>\n\nThe Government takes its responsib...,To mitigate data security risks on endpoint de...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer


In [220]:
evaluation_data = []
for index, row in test_df.iterrows():
    question = row.user_input
    ground_truth = row.reference

    retrieved_df = search_chunks(question)
    context_text = retrieved_df["chunk_text"].tolist()

    context = generate_context(retrieved_df)
    messages = generate_message(question, context)
    response = llm.invoke(messages)

    evaluation_data.append({
        "question": question,
        "answer": response.content,
        "contexts": context_text,
        "ground_truth": ground_truth
    })
eval_dataset = Dataset.from_list(evaluation_data)
    

    

    

In [223]:
df = eval_dataset.to_pandas()
df.iloc[9]

question        What are the technical measures used for digit...
answer          Based on the provided context, the technical m...
contexts        [Agencies should ensure that the digital water...
ground_truth    Technical measures and complementary process m...
Name: 9, dtype: object

In [229]:
config = RunConfig(
    max_workers=4, 
    timeout=120,
    max_retries=2
)
metrics = [
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
]
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=ragas_embeddings
)

Evaluating:  17%|█▋        | 8/48 [02:19<05:58,  8.96s/it]Exception raised in Job[8]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Evaluating:  67%|██████▋   | 32/48 [06:03<03:24, 12.80s/it]Exception raised in Job[27]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[31]: TimeoutError()
Exception raised in Job[32]: TimeoutError()
Evaluating: 100%|██████████| 48/48 [08:18<00:00, 10.38s/it]


In [230]:
print(results)

{'faithfulness': 0.8556, 'answer_relevancy': 0.9507, 'context_recall': 0.8750, 'context_precision': 1.0000}


In [231]:
results_df = results.to_pandas()

In [232]:
results_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision
0,What ar the goverment datasecurity policys?,[GOVERNMENT DATA SECURITY POLICIES This ...,The Government's data security policies are ou...,This document contains general information for...,NaN,0.865376,1.0,NaN
1,What are the key policies for government data ...,[Government Data Security Policies | 2\n\n\n\n...,"The key policies for government data security,...",The key policies for government data security ...,NaN,1.000000,1.0,NaN
2,What is the Government ICT Risk Management Met...,[\n\n| 0 ...,The Government ICT Risk Management Methodology...,The Government ICT Risk Management Methodology...,NaN,0.941476,1.0,NaN
3,What are the cybersecurity measures recommende...,[Government Data Security Policies | 8\n\n\n\n...,"Based on the provided context, the following c...",Agencies should implement the appropriate tech...,NaN,0.989787,0.5,NaN
4,What measures can be implemented to mitigate d...,[Section 3.2: Minimise the download of data to...,To mitigate data security risks on endpoint de...,To mitigate data security risks on endpoint de...,NaN,1.000000,0.0,NaN
5,What's the point of using hashing with salt?,[b) Partially hide the full data. Even if the ...,Hashing with salt is used to protect sensitive...,Hashing with salt is used to protect sensitive...,NaN,0.930653,1.0,NaN
6,What is the definition of a dataset and how do...,[| Terms |...,### Definition of a Dataset\nA dataset refers ...,A dataset refers to a collection of data which...,NaN,0.847900,1.0,NaN
7,How does tokenisation contribute to the Govern...,[\n\n| 0 ...,Tokenisation contributes to the Government ICT...,Tokenisation contributes to the Government ICT...,0.900000,1.000000,1.0,NaN
8,How can agencies protect data directly when it...,[b) Partially hide the full data. Even if the ...,Agencies can protect data directly when it is ...,Agencies can protect data directly by renderin...,NaN,0.883111,1.0,NaN
9,What are the technical measures used for digit...,[Agencies should ensure that the digital water...,"Based on the provided context, the technical m...",Technical measures and complementary process m...,NaN,0.976492,1.0,NaN


In [228]:
results_df.iloc[2]["retrieved_contexts"]

['\n\n| 0                                          | 1                                                                                                                                                                                                                                                                                                      |\n|:-------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|\n| Endpoint devices                           | Endpoint Devices refer to end-user devices designed for individual use (can be used by one or more users), Such as personal computers, mobile devices IP phones used to store, process or access Government data.                                                